# Embeddings dumper

This notebook serves to load a dataset of pairs of cell_ids, text and dump a .h5 file containing the pairs of cell_ids, embeddings

In [2]:
from text_embedder import TextProteinQueryEncoder
from protein_dataset import TextProteinQueryDataset
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import h5py
import numpy as np
import os

# Model & Dataset loading

In [3]:
# Loads the model
model_name = "nomic-ai/nomic-embed-text-v1"
encoder = TextProteinQueryEncoder(model_name)

Loading weights:   0%|          | 0/112 [00:00<?, ?it/s]

In [4]:
# Loads the dataset

dataset_name = "notebooks/cell_texts_augmented_K20_flip10.csv"

dataset = TextProteinQueryDataset(dataset_name, query_transform=lambda x: "search_query: "+x, in_memory=True)
loader = DataLoader(dataset, batch_size=256)

# Embeddings producing

## By chunks

In [ ]:
# Produces the embedding and saves chunks to disk to avoid out-of-memory errors
h5_path = f"{dataset_name}_embeddings.h5"

with torch.no_grad(), h5py.File(h5_path, "w") as f:
    dset_embs = None
    dset_ids = None

    for batch in tqdm(loader):
        queries, ids = batch
        emb = encoder(batch).cpu().numpy()
        id_array = np.array(ids, dtype=h5py.string_dtype())
        
        if dset_embs is None:
            # Initialize datasets with maxshape=None for the first dimension to allow resizing
            emb_dim = emb.shape[1]
            dset_embs = f.create_dataset("embeddings", shape=(0, emb_dim), maxshape=(None, emb_dim), dtype=emb.dtype, compression="gzip")
            dset_ids = f.create_dataset("cell_ids", shape=(0,), maxshape=(None,), dtype=h5py.string_dtype())
            
        # Resize datasets to accommodate the new batch
        current_size = dset_embs.shape[0]
        batch_size_actual = emb.shape[0]
        new_size = current_size + batch_size_actual
        
        dset_embs.resize(new_size, axis=0)
        dset_ids.resize(new_size, axis=0)
        
        # Append the new data
        dset_embs[current_size:new_size] = emb
        dset_ids[current_size:new_size] = id_array


 14%|█▍        | 1943/14125 [04:36<27:59,  7.25it/s]  

## All in one go

In [ ]:
# Produces the embedding

cell_ids = []
embeddings = []

with torch.no_grad():
    for batch in enumerate(tqdm(loader):
        queries, ids = batch
        emb = encoder(batch).cpu()
        
        cell_ids.extend(ids)
        embeddings.extend(emb)

In [ ]:
# Dumps into the h5 file

embeddings_tensor = torch.stack(embeddings).numpy()
id_array = np.array(cell_ids, dtype=h5py.string_dtype()) 

with h5py.File(f"{dataset_name}_embeddings.h5", "w") as f:
    f.create_dataset("embeddings", data=embeddings_tensor, compression="gzip")
    f.create_dataset("cell_ids", data=id_array)